In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from mdbmaster import MasterParams
from sys import prefix
mp = MasterParams(verbose=True)
io = FileIO()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from mdblib import genius
mio   = genius.MusicDBIO(verbose=True)
apiio = genius.RawAPIData()
db    = mio.db

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb
MusicDBBaseDirs(db=Genius)
   RawDataDir     = /Volumes/Piggy/Discog/artists-genius
   ModValDataDir  = /Volumes/Seagate/Discog/artists-genius-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-genius-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-genius
ParseRawDataUtils(mdbdata, mdbdir) [Genius]
Genius ModValMetaData
  ==> Basic
  ==> Media
  ==> Link
  ==> Metric
  ==> Counts


# Perm Dir

In [4]:
def setupPermDir(db):
    mp = MasterParams(verbose=False)
    prefixDir = DirInfo(prefix)
    projDir   = prefixDir.join(mp.getProjectName())
    projDir.mkDir()
    libDir    = projDir.join("mdblib")
    libDir.mkDir()
    permDBDir = libDir.join(db)
    permDBDir.mkDir()
    return permDBDir
permDBDir = setupPermDir(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Genius DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Genius


# Master Files

In [5]:
from mdbbase import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localAlbums        = MusicDBData(path=permDir, fname="{0}SearchedForLocalAlbums".format(db.lower()))
knownArtists       = mio.data.getArtistIDToNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [6]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Album Search:        {0}".format(len(localAlbums.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

Genius Search Results
   Global Master Search:      36804
   Local Artist Search:       207575
   Local Album Search:        165595
   Errors:                    277
   Known Summary IDs:         207575


In [ ]:
def createSearchedForAlbums():
    from fileutils import FileInfo
    known = {modVal: mio.dir.getRawAlbumModValDataDir(modVal).glob("*.p") for modVal in range(100)}
    albums = {modVal: {FileInfo(ifile).basename: True for ifile in files} for modVal, files in known.items()}
    albumnames = {modVal: {dbid: geniusKnownArtists.get(dbid) for dbid in dbids.keys()} for modVal,dbids in albums.items()}
    geniusLocalSearchedForAlbums = {}
    for modVal,dbids in albumnames.items():
        geniusLocalSearchedForAlbums.update(dbids)
    saveSearchedForLocalAlbums(geniusLocalSearchedForAlbums)

# Search For New Artists

In [7]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = genius.RawAPIData(debug=True)

Genius API(Key=lllWDHXkTwmxqpZCPyAA8EwX4pilPXKf7x4E_PKNDfMtiwtXvfahmVYL6WSb2mlQ)


## Find Artists To Get

# Download Album Data

In [8]:
mio   = genius.MusicDBIO(verbose=False,local=True,mkDirs=True)
apiio = genius.RawAPIData(debug=True)

Genius API(Key=lllWDHXkTwmxqpZCPyAA8EwX4pilPXKf7x4E_PKNDfMtiwtXvfahmVYL6WSb2mlQ)


## Find Albums To Get

In [15]:
print("Genius Search Results")
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(localAlbums.get().keys())]
artistNamesToGet = artistNamesToGet.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

Genius Search Results
   Known Summary IDs:    207575
   Artists IDs To Get:   11080


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "9:15pm")
#tt = TermTime("today", "9:15pm")
#tt = TermTime("today", "11:00am")
n  = 0
searchedForAlbums = localAlbums.get()
searchedForErrors = errors.get()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistSongs(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        errors.save(data=searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        localAlbums.save(data=searchedForAlbums)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
localAlbums.save(data=searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Starting Process [Getting Genius Albums]    ==> Time Is 2022-03-07 11:10:26
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-03-08 11:00:00 <====
   ====> Will Terminate Process 23 Hours and 49 Minutes From Now
Searching For Songs For Yung Bae (161492)                                 	   ===> 31  68
Searching For Songs For Tongo Hiti (2523788)                              	   ===> 3  3
Searching For Songs For Headhunterz & Crystal Lake (1665058)              	   ===> 1  1
Searching For Songs For Aytee (192416)                                    	   ===> 39 . 117
Searching For Songs For The Babies (31819)                                	   ===> 28  28
Searching For Songs For Schooner Fare (1428240)                           	   ===> 13  13
Searching For Songs For Mahler Chamber Orchestra (1991063)                	   ===> 2  2
Searching For Songs For Pepsi & Shirlie (365660)                          	   ===> 19  1

Searching For Songs For Shadowy Men On A Shadowy Planet (358572)          	   ===> 20  20
Searching For Songs For Gregory Kemmis (1935327)                          	   ===> 42  42
Searching For Songs For J. Michael Harter (257252)                        	   ===> 7  7
Searching For Songs For Mister Jackson (2588196)                          	   ===> 9  9
Searching For Songs For J Lindsay, Kiyanosh (2196619)                     	   ===> 4  4
Searching For Songs For Harry Nach, Kid Poison & Killua (2520567)         	   ===> 1  1
Searching For Songs For Corey Red (1572201)                               	   ===> 5  5
Searching For Songs For Something With Numbers (357354)                   	   ===> 40  40
Searching For Songs For Tristan Garel-Funk (1201360)                      	   ===> 41  41
Searching For Songs For Osama GBS (2323404)                               	   ===> 42  42
Searching For Songs For Scotty Boi, LTD3, & Graham (47271)                	   ===> 1  1
Searching For Songs Fo

Searching For Songs For Georgia Maq (252636)                              	   ===> 41  41
Searching For Songs For Damnation A.D. (374022)                           	   ===> 11  11
Searching For Songs For Shirish Swami (2758686)                           	   ===> 2  2
Searching For Songs For Krøllete (2426047)                                	   ===> 10  10
Searching For Songs For Jitterclick & Mikeshift (2919899)                 	   ===> 1  1
Searching For Songs For саша отис (sasha otis) (1042505)                  	   ===> 50  69
Searching For Songs For Dan Donegan (177598)                              	   ===> 49 . 139
Searching For Songs For Vacación (1919256)                               	   ===> 11  11
Searching For Songs For Belfa & Gina Livia (2561300)                      	   ===> 2  2
Searching For Songs For Ale Zuliani (1504556)                             	   ===> 1  1
Searching For Songs For KAGY SYSTEME (565320)                             	   ===> 3  3
Searching For Song

Searching For Songs For Kaskade & Felix Cartal (671527)                   	   ===> 3  3
Searching For Songs For KEVVO, Myke Towers & Darell (2076587)             	   ===> 1  1
Searching For Songs For Jmb Jordan (2609913)                              	   ===> 2  2
Searching For Songs For Souls (355379)                                    	   ===> 43  46
Searching For Songs For SLS Music (1708504)                               	   ===> 36 ... 163
Searching For Songs For Lella Fadda (1634103)                             	   ===> 9  9
Searching For Songs For Tone Onna Mission (2776666)                       	   ===> 1  1
Searching For Songs For Colourbox (368478)                                	   ===> 26  26
Searching For Songs For Basil Rathbone (460217)                           	   ===> 1  1
Searching For Songs For Gillzer (1368757)                                 	   ===> 4  4
Searching For Songs For Richard Tognetti,Australian Chamber Orchestra (521672)	   ===> 6  6
Searching For Song

Searching For Songs For sick to the back teeth (1923454)                  	   ===> 50  55
Searching For Songs For Daniela Andrade (452357)                          	   ===> 47  81
Searching For Songs For Lange Ritch (960604)                              	   ===> 16  16
Searching For Songs For John Littlejohn (2230663)                         	   ===> 1  1
Searching For Songs For Imaginary Friends (1647797)                       	   ===> 13  13
Searching For Songs For Berliner Symphoniker,Karin Lechner (521631)       	   ===> 1  1
Searching For Songs For Cxdy (1054484)                                    	   ===> 47 ...................... 1123
Searching For Songs For Lilessay (1919312)                                	   ===> 14  14
Searching For Songs For Curt Cress (1028130)                              	   ===> 49  52
Searching For Songs For Fat Soldiers (243805)                             	   ===> 50  62
Searching For Songs For Rambutan Jam Band (1866173)                       	   ==

...

Searching For Songs For L'Morphine (309616)                               	   ===> 50 ... 205
Searching For Songs For Hear 'N Aid (386896)                              	   ===> 1  1
6050/?     : Process [Getting Genius Albums] Has Run For 8 Hours and 50 Minutes.
Saving 202545 Genius Searched For Albums
Searching For Songs For Bomayé Musik (1033386)                           	   ===> 49 .. 169
Searching For Songs For Chester Bennington (1015)                         	   ===> 50 ....... 406
Searching For Songs For Chencho Corleone, Wisin & Natti Natasha (2051691) 	Error ==> ('2051691', 'Chencho Corleone, Wisin & Natti Natasha')
Searching For Songs For Carlos Alomar (646155)                            	   ===> 50 . 125
Searching For Songs For Greg Raposo (378908)                              	   ===> 11  11
Searching For Songs For MCK (156641)                                      	   ===> 27  49
Searching For Songs For Pops Fernandez (377761)                           	   ===> 5  5
Searc

Searching For Songs For Gabriel Drago (2009653)                           	   ===> 15  15
Searching For Songs For Djakac (1900001)                                  	   ===> 1  1
Searching For Songs For Whethan (990491)                                  	   ===> 50 .. 178
6125/?     : Process [Getting Genius Albums] Has Run For 8 Hours and 57 Minutes.
Saving 202620 Genius Searched For Albums
Searching For Songs For 1919 (1758724)                                    	   ===> 15  15
Searching For Songs For Dadá Boladão (1258523)                          	   ===> 13  13
Searching For Songs For Sharaktah (1621504)                               	   ===> 13  13
Searching For Songs For 7Bantaiz & D'EVIL (2251998)                       	   ===> 1  1
Searching For Songs For Lowertown (1714994)                               	   ===> 24  24
Searching For Songs For Sean Brown (12815)                                	   ===> 24 . 60
Searching For Songs For Bajaga (352118)                             

Searching For Songs For Iuvenes (356082)                                  	   ===> 3  3
6200/?     : Process [Getting Genius Albums] Has Run For 9 Hours and 4 Minutes.
Saving 202695 Genius Searched For Albums
Searching For Songs For Dostrescinco (1751097)                            	   ===> 36  39
Searching For Songs For Virgínia Rodrigues (1296254)                     	   ===> 2  2
Searching For Songs For Peter Lind Hayes & Mary Healy (1993343)           	   ===> 4  4
Searching For Songs For REVOLT (84021)                                    	   ===> 9  9
Searching For Songs For Atlantic Records (68978)                          	   ===> 50 ............................................................................................................................ 6200
Searching For Songs For Gazda Paja (244347)                               	   ===> 40  52
Searching For Songs For Rothko (2606207)                                  	   ===> 0  0
Searching For Songs For Barna synger (45419

Searching For Songs For Florence Churchill Casebeer (40004)               	   ===> 26  26
6275/?     : Process [Getting Genius Albums] Has Run For 9 Hours and 21 Minutes.
Saving 202770 Genius Searched For Albums
Searching For Songs For Chris Mars (33397)                                	   ===> 49  55
Searching For Songs For Katy X (623492)                                   	   ===> 4  4
Searching For Songs For Snails & Krimer (1842704)                         	   ===> 3  3
Searching For Songs For Dancermusicc (2937033)                            	   ===> 7  7
Searching For Songs For Busted (158503)                                   	   ===> 50  80
Searching For Songs For Lee Walker & DJ Deeon (766671)                    	   ===> 1  1
Searching For Songs For VEYasin (1177547)                                 	   ===> 50  82
Searching For Songs For Sebastian Kamae (1410661)                         	   ===> 25  25
Searching For Songs For Bow Wow & Omarion (18995)                         	 

Searching For Songs For Sean Slick (632082)                               	   ===> 50  50
Searching For Songs For Omeretta the Great (1061799)                      	   ===> 35  37
Searching For Songs For mrld (2563374)                                    	   ===> 8  8
Searching For Songs For Paul Grønseth (529935)                            	   ===> 4  4
Searching For Songs For Frijo (1184581)                                   	   ===> 49 . 129
Searching For Songs For Josh Garrels (43072)                              	   ===> 44 .. 153
Searching For Songs For GARYSHAWN (995006)                                	   ===> 26  26
Searching For Songs For Josi-F AkA KatzBeatz (298919)                     	   ===> 1  1
Searching For Songs For William Byrd (1600611)                            	   ===> 24  24
Searching For Songs For Rasha_Megastar (1197027)                          	   ===> 25  25
Searching For Songs For Vala - The Female Rapper (42682)                  	   ===> 1  1
Searching For

# Move Local Files

In [ ]:
from mdblib.genius import moveLocalFiles
moveLocalFiles()

In [ ]:
from fileutils import FileInfo
from mdbmaster import MasterParams
mp        = MasterParams()
mioLocal  = genius.MusicDBIO(local=True,mkDirs=True,debug=True)
mioGlobal = genius.MusicDBIO(local=False,mkDirs=True,debug=True)
for modVal in mp.getModVals():
    files = mioLocal.dir.getRawAlbumModValDataDir(modVal).glob("*.p")
    _ = [FileInfo(ifile).mvFile(FileInfo(mioGlobal.data.getRawArtistAlbumFilename(modVal,ifile.stem))) for ifile in files]

# Parse What We Got

In [ ]:
#prd = ParseRawData(mio.data, mio.dir, verbose=False)
%autoreload
from mdbutils import poolIO
mio = genius.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(3,100))

# Inf

In [ ]:
%autoreload
from masterManualEntriesUtils import masterManualEntriesUtils
from masterArtistNameCorrection import masterArtistNameCorrection
from ioUtils import fileIO
from pandas import Series

meu   = masterManualEntriesUtils()
mmeDF = meu.getDF()
manc  = masterArtistNameCorrection()
io    = fileIO()
knownArtistNames = mmeDF["ArtistName"].apply(manc.clean).apply(lambda x: x[:245])
print("Found {0} Artists".format(knownArtistNames.shape[0]))
searchedForArtists = Series(io.get("lastfmSearchedForArtists.p"))
print("Found {0} Searched For Artists".format(len(searchedForArtists)))
artistNames = knownArtistNames[~knownArtistNames.index.isin(searchedForArtists.index)].copy(deep=True)
print("Found {0} Artists - Searched For".format(artistNames.shape[0]))

In [ ]:
%autoreload
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("LastFM")
api  = apig.getDBAPIData("LastFM")

searchedForArtists = io.get("lastfmSearchedForArtists.p")
errs = io.get("lastfmErrorsSearchedForArtists.p")
ts = timestat("Downloading LastFM Data")
tt = termTime("today", "9:30pm")
n  = 0
for i,(idx,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(idx) is not None:
        continue
    if errs.get(idx) is not None:
        continue
    dbID = dbIO.getdbID(artistName)
    if not dbIO.isArtistInfoKnown(dbID) or True:
        response = api.getArtistInfo(artistName)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((idx,artistName)))
        errs[idx] = artistName
        io.save(idata=errs, ifile="lastfmErrorsSearchedForArtists.p")
        sleep(5)
        continue
    dbIO.saveArtistInfo(dbID, response)
    searchedForArtists[idx] = artistName
    api.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        api.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("="*150)
        print("Saving {0} LastFM Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="lastfmSearchedForArtists.p")
        api.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} LastFM Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="lastfmSearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} LastFM Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="lastfmErrorsSearchedForArtists.p")

In [ ]:
from artistGeniusAPI import artistGeniusAPI
from dbBase import dbBase

from fsUtils import dirUtil,fileUtil
from artistIDBase import dbArtistID
from artistModValue import artistModValue


##################################################################################################################
# Base Class
##################################################################################################################
class dbGeniusAPI:
    def __init__(self):
        self.db     = "GeniusAPI"
        self.disc   = dbBase(self.db.lower())
        self.artist = artistGeniusAPI(self.disc)
        self.aid    = dbArtistID(self.db)

        self.getpsid   = self.aid.getpsid
        self.getModVal = self.aid.getModVal
        
        self.io = fileIO()
        
        
    def getArtistInfoSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        return fileUtil(modValDir).join("{0}.p".format(artistID))
            
    def saveArtistInfo(self, artistID, artistInfo):
        self.io.save(idata=artistInfo, ifile=self.getArtistInfoSavename(artistID))
        
        
    def getArtistAlbumsSavename(self, artistID):
        modValDir = dirUtil(self.disc.getArtistsDir()).join(str(self.getModVal(artistID)))
        albumsDir = dirUtil(modValDir()).join("albums")
        return fileUtil(albumsDir).join("{0}.p".format(artistID))
    
    def saveArtistAlbums(self, artistID, artistAlbums):
        self.io.save(idata=artistAlbums, ifile=self.getArtistAlbumsSavename(artistID))

In [ ]:
import requests
from time import sleep

class apiIO:
    def __init__(self, name):
        self.name = name
        self.code = None
        self.url = None
        self.response = None
        
    def get(self, url):
        self.url       = url
        try:
            self.response  = requests.get(url)
        except:
            self.response  = None
            self.code      = 0
            return {}
            
        self.code      = self.response.status_code
        
        try:
            json_data      = self.response.json() if self.response.status_code == 200 else {}
        except:
            json_data      = {}
        return json_data
    
    def getResponse(self):
        return self.response
    
    def getURL(self):
        return self.url
    
    def getStatus(self):
        return self.code
    
    def sleep(self, value):
        sleep(value)

In [ ]:
from urllib.parse import quote
    
    
class geniusAPIIO(apiIO):
    def __init__(self):
        super().__init__("Genius")
        client_access_token = "lllWDHXkTwmxqpZCPyAA8EwX4pilPXKf7x4E_PKNDfMtiwtXvfahmVYL6WSb2mlQ"
        self.apikey = client_access_token

        
        self.baseURL = "http://api.genius.com"
        self.format  = "json"
        self.options = {"per_page": "50"}
        self.method  = {"TopTracks": "artist.gettoptracks", "TopAlbums": "artist.gettopalbums", "ArtistInfo": "artist.getinfo", "ArtistSearch": "artist.search"}
        
        print("{0} API(Key={1})".format(self.name, self.apikey))
        
        #requestURL = "http://ws.audioscrobbler.com/2.0/?method=artist.gettoptracks&artist={0}&api_key={1}&limit=10000&format=json".format(quote(artistName), self.apikey)
        #requestURL = "http://ws.audioscrobbler.com/2.0/?method=artist.gettopalbums&artist={0}&api_key={1}&limit=10000&format=json".format(quote(artistName), self.apikey)
        #requestURL = "https://ws.audioscrobbler.com/2.0/?method=artist.getinfo&artist={0}&api_key={1}&format=json".format(quote(artistName), self.apikey)
        #requestURL = "https://ws.audioscrobbler.com/2.0/?method=artist.search&artist={0}&api_key={1}&format=json".format(quote(artistName), self.apikey)

        #genius_artist_url = f"http://api.genius.com/artists/{artist_id}?access_token={self.apikey}"
    ##################################################################################################################################################################
    # API Parser
    ##################################################################################################################################################################
    def getResponse(self, response):
        retval = response.get('response', {}) if isinstance(response, dict) else {}
        return retval

    
    ##################################################################################################################################################################
    # Artist Info
    ##################################################################################################################################################################
    def getArtistInfoURL(self, artist_id):
        return "{0}/artists/{1}?access_token={2}".format(self.baseURL, artist_id, self.apikey)
    
    def getArtistInfo(self, artistName, artistID):
        print("Searching For Songs For {0: <50}\t".format("{0} ({1})".format(artistName,artistID)), end="")
        geniusRecord = self.getResponse(self.get(self.getArtistInfoURL(artistID)))
        print(" {0}".format(len(geniusRecord)))
        return geniusRecord

        
    ##################################################################################################################################################################
    # Artist Search
    ##################################################################################################################################################################
    def getArtistSearchURL(self, search_term):
        #genius_search_url = f"http://api.genius.com/search?q={search_term}&access_token={client_access_token}"
        return "{0}/search?{1}&access_token={2}".format(self.baseURL, search_term, self.apikey)
        #return genius_search_url

    def getArtistSearch(search_term):
        response = self.get(self.getArtistSearchURL(search_term))
        results  = response.get('response', {}) if isinstance(response, dict) else {}
        hits     = results.get('hits', [])

        geniusRecords = Series([geniusSearchRecord(item).get() for item in hits], dtype = 'object')    
        nUnique = geniusRecords.apply(lambda x: x['artist']['id']).nunique()
        print(" {0}/{1}".format(nUnique,len(geniusRecords)))
        
        return geniusRecords


    ##################################################################################################################################################################
    # Artist Info
    ##################################################################################################################################################################
    def getArtistSongsURL(self, artist_id, page, per_page=50):
        #genius_artist_songs_url = f"http://api.genius.com/artists/{artist_id}/songs?per_page={per_page}?&page={page}&access_token={client_access_token}"
        return "{0}/artists/{1}/songs?per_page={2}&page={3}&access_token={4}".format(self.baseURL, artist_id, self.options["per_page"], page, self.apikey)
        #return genius_artist_songs_url    
        
    def getArtistSongs(self, artistName, artistID):
        print("Searching For Songs For {0: <50}\t".format("{0} ({1})".format(artistName,artistID)), end="")
        searchResults  = []
        page           = 1
        requestResult  = self.getResponse(self.get(self.getArtistSongsURL(artistID, page=page)))
        if len(requestResult) == 0:
            return None
        page = requestResult.get('next_page', None)
        try:
            searchResults += requestResult['songs']
        except:
            return None
        print("   ===> {0}".format(len(searchResults)), end=" ")
        while page is not None:
            self.api.sleep(2.0)
            requestResult  = self.getResponse(self.get(self.getArtistSongsURL(artistID, page=page)))
            try:
                searchResults += requestResult['songs']
            except:
                break
            page = requestResult.get('next_page', None)
            if page:
                #print("{0}".format(len(searchResults)), end="")
                print(".", end="")
        print(" {0}".format(len(searchResults)))

        albums = [geniusAlbumsRecord(album).get() for album in searchResults]
        retval = {'artistID': artistID, 'albums': albums}
        return retval
        


    ##################################################################################################################################################################
    # Artist Search
    ##################################################################################################################################################################
    def getArtistSearchURL(self, artistName):
        return "{0}?method={1}&artist={2}&api_key={3}&format={4}".format(self.baseURL, self.method["ArtistSearch"], quote(artistName), self.apikey, self.format)
    
    def getArtistSearch(self, artistName, show=True):
        print("Searching For {0: <50}".format(artistName), end="")
        response = self.get(self.getArtistSearchURL(artistName))
        results  = response.get('response', {}) if isinstance(response, dict) else {}
        hits     = results.get('hits', [])

        geniusRecords = Series([geniusSearchRecord(item).get() for item in hits], dtype = 'object')    
        nUnique = geniusRecords.apply(lambda x: x['artist']['id']).nunique()
        print(" {0}/{1}".format(nUnique,len(geniusRecords)))
        
        return geniusRecords

# API Data

In [ ]:
search_term = "Missy Elliott"

In [ ]:
stop = 5000
dbIO = dbGeniusAPI()
api  = geniusAPIIO()
ts   = timestat("Getting Artist Data From Genius API")
n    = 0

In [ ]:
from glob import glob
from masterUtils import masterUtils
from fsUtils import dirUtil
mu = masterUtils()
artistsDir = dirUtil(mu.getDisc("GeniusAPI").getArtistsDir())
#### This takes forever...
#geniusArtistFiles = {modVal: glob(str(dirUtil(artistsDir.join(str(modVal))).join("*.p"))) for modVal in range(100)}

# Download Genius Artist Data

## Determine Artists To Download

In [ ]:
from masterUtils import masterUtils
from masterArtistNameCorrection import masterArtistNameCorrection
from dbIOGate import dbIOGate
gate = dbIOGate()
dbIO = gate.getDBIO("Genius")

artistsDF = io.get("geniusArtistRanking.p")
artistsDF.index = [str(x) for x in artistsDF.index]
print("Found {0} Ranked Artists".format(artistsDF.shape[0]))
artistIDToNameData = dbIO.data.getArtistIDToNameData()
print("Found {0} Known Artists".format(artistIDToNameData.shape[0]))
geniusIDsToDownload = artistsDF[~artistsDF.index.isin(artistIDToNameData.index)]
artistNames = geniusIDsToDownload['name']
print("Found {0} Artists To Download".format(artistNames.shape[0]))

## Download Artist Info Data

In [ ]:
%autoreload
from timeUtils import timestat, termTime

dbIO = dbg.getDBIO("Genius")
api  = apig.getDBAPIData("Genius")

searchedForArtists = io.get("geniusSearchedForArtists.p")
errs = io.get("geniusErrorsSearchedForArtists.p")
ts = timestat("Downloading Genius Data")
#tt = termTime("today", "9:00pm")
tt = termTime("tomorrow", "11:00am")
#tt = termTime("today", "9:00am")
n  = 0
for i,(dbID,artistName) in enumerate(artistNames.iteritems()):    
    if searchedForArtists.get(dbID) is not None:
        continue
    if errs.get(dbID) is not None:
        continue
    if not dbIO.isArtistInfoKnown(dbID) or True:
        response = api.getArtistInfo(artistName, dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        errs[dbID] = artistName
        io.save(idata=errs, ifile="geniusErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistInfo(dbID, response)
    searchedForArtists[dbID] = artistName
    api.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        api.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} Genius Searched For Artists".format(len(searchedForArtists)))
        io.save(idata=searchedForArtists, ifile="geniusSearchedForArtists.p")
        print("="*150)
        api.sleep(2.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} Genius Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="geniusSearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Genius Errors For Artists".format(len(errs)))
    io.save(idata=errs, ifile="geniusErrorsSearchedForArtists.p")

In [ ]:
dbID,artistName = ('1122189', 'ELVIS$') #('1076825', 'Mid-Cities Worship')
errs[dbID] = artistName
io.save(idata=errs, ifile="geniusErrorsSearchedForArtists.p")

In [ ]:

print("Saving {0} Genius Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="geniusSearchedForArtists.p")

In [ ]:
stop = 10000
dbIO = dbGeniusAPI()
api  = geniusAPIIO()
ts   = timestat("Getting Artist Data From Genius API")
n    = 0


searchedForArtists = io.get("geniusSearchedForArtists.p")
errs = io.get("geniusErrorsSearchedForArtists.p")
for i,(artistID,artistName) in enumerate(artistNames.iteritems()):
    if searchedForArtists.get(artistID) is not None:
        continue
    if errs.get(artistID) is not None:
        continue
    if n > stop:
        break
        
    savename = dbIO.getArtistInfoSavename(artistID)
    if savename.exists():
        print("Known ==> {0}".format((artistID,artistName,savename)))
        searchedForArtists[artistID] = artistName 
        continue
        
    response = api.getArtistInfo(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        errs[artistID] = artistName
        io.save(idata=errs, ifile="geniusErrorsSearchedForArtists.p")
        api.sleep(5)
        continue
    dbIO.saveArtistInfo(artistID=artistID, artistInfo=response)
    searchedForArtists[artistID] = artistName
    api.sleep(2.5)
    n += 1
    
    if n % 25 == 0:
        api.sleep(2.5)
        print("="*150)
        ts.update(n=n,N=stop)
        print("="*150)
        io.save(idata=searchedForArtists, ifile="geniusSearchedForArtists.p")
        api.sleep(2.5)
    
ts.stop()
print("Saving {0} Genius Searched For Artists".format(len(searchedForArtists)))
io.save(idata=searchedForArtists, ifile="geniusSearchedForArtists.p")
if len(errs) > 0:
    print("Saving {0} Genius Searched For Artists".format(len(errs)))
    io.save(idata=errs, ifile="geniusErrorsSearchedForArtists.p")

# Download Genius URLs

In [ ]:
from dbArtistsGenius import dbArtistsGenius

In [ ]:
#https://www.jiosaavn.com/song/terrestrial/HRovUgFpbl4


url= "https://genius.com/artists/Sanari"
url= "https://genius.com/artists/Kira-jp"
url= "https://genius.com/artists/Joipe"
url= "https://genius.com/artists/Aron-can"
url= "https://genius.com/artists/Herra-hnetusmjor"
url= "https://genius.com/artists/Omer-adam"
urls=[]
from time import sleep
#urls.append("https://genius.com/artists/Beastie-boys")
urls.append("https://genius.com/artists/Saske")
urls.append("https://genius.com/artists/Fy")
urls.append("https://genius.com/artists/Eden-ben-zaken")
urls.append("https://genius.com/artists/Itay-levi")
urls.append("https://genius.com/artists/Snik")
urls.append("https://genius.com/artists/Joey-christ")
urls=[] 
urls.append("https://genius.com/artists/Canozan")
urls.append("https://genius.com/artists/Kaya-giray")
urls.append("https://genius.com/yungouzo")
urls.append("https://genius.com/artists/Nahide-babashl")
urls.append("https://genius.com/artists/Brado")
urls.append("https://genius.com/artists/Narkoz")
urls.append("https://genius.com/artists/Gunay-aksoy")
urls=[]
urls.append("https://genius.com/artists/Vallis-alps")
urls.append("https://genius.com/artists/G-flip")
urls.append("https://genius.com/artists/Chasing-abbey")
urls.append("https://genius.com/artists/Hp-boyz")
urls.append("https://genius.com/artists/Youngn-lipz")
urls.append("https://genius.com/artists/No-money-enterprise")
urls.append("https://genius.com/artists/Louisa")
urls.append("https://genius.com/artists/Sigma")
urls.append("https://genius.com/artists/Cxloe")
urls.append("https://genius.com/artists/Mallrat")
urls.append("https://genius.com/artists/Tyron-hapi")
urls.append("https://genius.com/artists/Chillinit")
urls.append("https://genius.com/artists/Hooligan-hefs")
urls.append("https://genius.com/artists/Onefour")
urls=["https://genius.com/albums/Mudi/Amal", 
"https://genius.com/artists/I-waal", 
"https://genius.com/artists/Colin-firth", 
"https://genius.com/artists/5-after-midnight", 
"https://genius.com/artists/Sunset-bros-x-mark-mccabe", 
"https://genius.com/artists/Migos", 
"https://genius.com/artists/The-academic", 
"https://genius.com/artists/Josh-dylan", 
"https://genius.com/artists/Hugh-skinner", 
"https://genius.com/artists/Jeremy-irvine", 
"https://genius.com/artists/Offaiah", 
"https://genius.com/artists/The-crystals", 
"https://genius.com/artists/Jeremy-soule", 
"https://genius.com/artists/Tiagz", 
"https://genius.com/artists/Lil-xxel", 
"https://genius.com/artists/Irish-women-in-harmony", 
"https://genius.com/artists/Robert-grace", 
"https://genius.com/artists/Ryan-blyth", 
"https://genius.com/artists/Whtkd", 
"https://genius.com/artists/Pnl", 
"https://genius.com/artists/Slumberjack", 
"https://genius.com/artists/Passion", 
"https://genius.com/artists/Biggy", 
"https://genius.com/artists/Pon-cho", 
"https://genius.com/artists/Run-dmc", 
"https://genius.com/artists/Golden-features", 
"https://genius.com/artists/Dastic", 
"https://genius.com/artists/Kita-alexander", 
"https://genius.com/artists/Stace-cadet", 
"https://genius.com/artists/Carmouflage-rose", 
"https://genius.com/artists/Fergus-james", 
"https://genius.com/artists/Triple-one", 
"https://genius.com/artists/Day1", 
"https://genius.com/artists/Lyra", 
"https://genius.com/artists/Af1"]
urls=["https://genius.com/artists/Liga"]
urls=["https://genius.com/artists/Le-mo-dnk", 'https://genius.com/artists/Benee']
urls=["https://genius.com/artists/Passion-dnk", 'https://genius.com/artists/One-bit-and-noah-cyrus', 'https://genius.com/artists/Soule']
urls=["https://genius.com/artists/Rin", "https://genius.com/artists/Fler", "https://genius.com/artists/Gringo",
"https://genius.com/artists/Lil-lano", "https://genius.com/artists/Shirin-david", "https://genius.com/artists/Monet192",
"https://genius.com/artists/Kasimir1441", "https://genius.com/artists/Badmomzjay", "https://genius.com/artists/Jazn",
"https://genius.com/artists/Reezy", "https://genius.com/artists/King-khalil", "https://genius.com/artists/Celine",
"https://genius.com/artists/Bhz", "https://genius.com/artists/Xatar", "https://genius.com/artists/Finch-asozial",
"https://genius.com/artists/Hanybal", "https://genius.com/artists/Tj-beastboy", "https://genius.com/artists/Abdi",
"https://genius.com/artists/Fourty", "https://genius.com/artists/Sdp", "https://genius.com/artists/Ngee",
"https://genius.com/artists/Sinan-g", "https://genius.com/artists/Rais", "https://genius.com/artists/Sxtn",
"https://genius.com/artists/Ali471", "https://genius.com/artists/Reynmen", "https://genius.com/artists/Bozza",
"https://genius.com/artists/Metrickz", "https://genius.com/artists/Slavik", "https://genius.com/artists/Play69",
"https://genius.com/artists/Spongebozz", "https://genius.com/artists/Juri", "https://genius.com/artists/Saaftig",
"https://genius.com/artists/Mois", "https://genius.com/artists/Dhurata-dora", "https://genius.com/artists/Knossi","https://genius.com/artists/Gent"]
urls=["https://genius.com/artists/Casper", "https://genius.com/artists/Yonii", "https://genius.com/artists/Belah",
"https://genius.com/artists/Kayef", "https://genius.com/artists/Elif", "https://genius.com/artists/Deno419",
"https://genius.com/artists/Camila-cabello", "https://genius.com/artists/Garagen-larrys", "https://genius.com/artists/Beyazz",
"https://genius.com/artists/Firat", "https://genius.com/artists/Hitimpulse", "https://genius.com/artists/Sarhad",
"https://genius.com/artists/Ivana-santacruz", "https://genius.com/artists/Schubi-akpella", "https://genius.com/artists/Soufian",
"https://genius.com/artists/Kilomatik", "https://genius.com/artists/Tarek-kiz", "https://genius.com/artists/Tayna",
"https://genius.com/artists/Die-schwarzwalder-kirschtorten", "https://genius.com/artists/Gary-washington", "https://genius.com/artists/Whethan",
"https://genius.com/artists/Anstandslos-and-durchgeknallt", "https://genius.com/artists/Jay-samuelz", "https://genius.com/artists/Sugar-mmfk",
"https://genius.com/artists/Delil", "https://genius.com/artists/Apored", "https://genius.com/artists/Payy",
"https://genius.com/artists/Joshi-mizu", "https://genius.com/artists/Mike-singer", "https://genius.com/artists/Naaz",
"https://genius.com/artists/Luqe", "https://genius.com/artists/Laruzo", "https://genius.com/artists/Sarah-lombardi",
"https://genius.com/artists/Tom-gregory", "https://genius.com/artists/Fake-pictures", "https://genius.com/artists/Ghetto-phenomene"]
urls=["https://genius.com/artists/Dante-yn", "https://genius.com/artists/Reda-rwena", "https://genius.com/artists/Remoe",
     "https://genius.com/artists/Lucky-luke", "https://genius.com/artists/Paula-dalla-corte", "https://genius.com/artists/Inoffiziellgoldenboy",
     "https://genius.com/artists/Melo68", "https://genius.com/artists/Casar", "https://genius.com/artists/Aylo",
     "https://genius.com/artists/Lune", "https://genius.com/artists/Phil-the-beat", "https://genius.com/artists/Amanda-delara",
     "https://genius.com/artists/Monk", "https://genius.com/artists/Thedodo", "https://genius.com/artists/Harris-and-ford",
     "https://genius.com/artists/Kidda", "https://genius.com/artists/Georg-stengel", "https://genius.com/artists/Qzeng",
     "https://genius.com/artists/Hugel", "https://genius.com/artists/Querbeat", "https://genius.com/artists/Kynda-gray",
     "https://genius.com/artists/Estikay", "https://genius.com/artists/Hemso", "https://genius.com/artists/Patron",
     "https://genius.com/artists/Raf-camora", "https://genius.com/artists/Undacava", "https://genius.com/artists/Jiggo"]
urls=["https://genius.com/artists/Yung-kafa-and-kucuk-efendi"]
urls=["https://genius.com/artists/Powerkryner", "https://genius.com/artists/Stupid-goldfish", "https://genius.com/artists/Yassazin",
 "https://genius.com/artists/Avec", "https://genius.com/artists/Anastasija", "https://genius.com/artists/Kyd-the-band",
 "https://genius.com/artists/Stefan-rauch", "https://genius.com/artists/Ahmad-amin", "https://genius.com/artists/Masn",
 "https://genius.com/artists/Greeen", "https://genius.com/artists/Mc-stojan", "https://genius.com/artists/Alle-achtung",
 "https://genius.com/artists/Devito", "https://genius.com/artists/Relja", "https://genius.com/artists/Chris-steger",
 "https://genius.com/artists/Nazar", "https://genius.com/artists/Sultan"]
urls=['https://genius.com/artists/Nightcall', 'https://genius.com/artists/Darius-and-finlay']
urls=['https://genius.com/artists/Fy', 'https://genius.com/artists/Snik', 'https://genius.com/artists/Rap-viet',
     'https://genius.com/artists/Mad-clip', 'https://genius.com/artists/Moira-dela-torre', 'https://genius.com/artists/Lala-hsu',
     'https://genius.com/artists/Sin-boy', 'https://genius.com/artists/My-tam', 'https://genius.com/artists/Eladio-carrion',
     'https://genius.com/artists/Son-tung-m-tp', 'https://genius.com/artists/The-toys', 'https://genius.com/artists/The-toys-thai-artist',
     'https://genius.com/artists/Eric-chou', 'https://genius.com/artists/Dear-jane', 'https://genius.com/artists/Illeoo',
     'https://genius.com/artists/Junoflo', 'https://genius.com/artists/En', 'https://genius.com/artists/Hebe-tien',
     'https://genius.com/artists/Fer-palacio', 'https://genius.com/artists/Payung-teduh', 'https://genius.com/artists/Ysy-a',
     'https://genius.com/artists/Miss-ko', 'https://genius.com/artists/Dj-alex-arg', 'https://genius.com/artists/Bnk48',
     'https://genius.com/artists/Vu-cat-tuong', 'https://genius.com/artists/Accusefive', 'https://genius.com/artists/Mj116',
     'https://genius.com/artists/Nine-chen', 'https://genius.com/artists/Getsunova', 'https://genius.com/artists/Earth-patravee',
     'https://genius.com/artists/Alien-huang', 'https://genius.com/artists/Endy-chow', 'https://genius.com/artists/Singto-numchok',
     'https://genius.com/artists/Zi-stefan-chen', 'https://genius.com/artists/Oat-pramote', 'https://genius.com/artists/Ngot',
     'https://genius.com/artists/Bird-thongchai', 'https://genius.com/artists/Leowang', 'https://genius.com/artists/Kelly-yu-wen-wen',
     'https://genius.com/artists/Noo-phuoc-thinh', 'https://genius.com/artists/Zom-marie', 'https://genius.com/artists/Og-anic',
     'https://genius.com/artists/Maja-salvador-and-tor-saksit', 'https://genius.com/artists/Meyou', 'https://genius.com/artists/Cocktail']
urls=['https://genius.com/artists/Diskoria', 'https://genius.com/DJ_OOPs', 'https://genius.com/artists/Tugba-yurt',
     'https://genius.com/artists/Lost-sky', 'https://genius.com/artists/Dawn-oberg', 'https://genius.com/artists/Cengiz-ates',
     'https://genius.com/artists/Gal-adam', 'https://genius.com/artists/Bulat-nurimov', 'https://genius.com/artists/Taypan',
     'https://genius.com/artists/Jay-fendi-prince-of-lawtown', 'https://genius.com/artists/Ahmet-safak',
     'https://genius.com/artists/Hipsumhaps', 'https://genius.com/artists/Orri', 'https://genius.com/artists/Wonayd',
     'https://genius.com/artists/Vremya-i-steklo', 'https://genius.com/artists/Fadel-chaker', 'https://genius.com/artists/I61',
     'https://genius.com/artists/Matti-rssland', 'https://genius.com/artists/Vtornik', 'https://genius.com/artists/Helgi-bjornsson',
     'https://genius.com/artists/Herra-ylppo', 'https://genius.com/artists/Sebastian-wallden', 'https://genius.com/artists/Joker-xue',
     'https://genius.com/artists/Rkm-and-ken-y', 'https://genius.com/artists/John-mcdermott', 'https://genius.com/artists/Ronan-tynan',
     'https://genius.com/artists/Timmy-xu', 'https://genius.com/artists/John-p-kee', 'https://genius.com/artists/John-p-kee-and-the-new-life-community-choir',
     'https://genius.com/artists/Ilegales', 'https://genius.com/artists/James-wilson', 'https://genius.com/artists/Millie-b',
     'https://genius.com/artists/Stephen-paul-taylor', 'https://genius.com/artists/Byrne-and-kelly', 'https://genius.com/artists/Consequence',
     'https://genius.com/artists/Xtreme', 'https://genius.com/artists/Hua-chenyu', 'https://genius.com/artists/Bob-james',
     'https://genius.com/artists/Niska', 'https://genius.com/artists/Loon', 'https://genius.com/artists/Morgenshtern-and-eldzhey',
     'https://genius.com/artists/Morgenshtern', 'https://genius.com/artists/Jessie-morales', 'https://genius.com/artists/Jessie-morales-el-original-de-la-sierra']
urls=['https://genius.com/artists/Casino', 'https://genius.com/artists/Mark-harris', 'https://genius.com/artists/Schoolboy-q',
     'https://genius.com/artists/Hollis', 'https://genius.com/artists/Skipper', 'https://genius.com/artists/Trillville',
     'https://genius.com/artists/Cutty', 'https://genius.com/artists/Paul-mccoy']
urls=['https://genius.com/artists/Opshop', 'https://genius.com/artists/Elias', 'https://genius.com/artists/In-hearts-wake',
     'https://genius.com/artists/Kash-doll', 'https://genius.com/artists/Hov1', 'https://genius.com/artists/Smashproof',
     'https://genius.com/KuZ', 'https://genius.com/artists/Pannirselvam']
urls=['https://genius.com/artists/Helen-corry', 'https://genius.com/artists/Jetski-safari', 'https://genius.com/artists/Guccihighwaters',
     'https://genius.com/artists/Gmx', 'https://genius.com/artists/Liran-danino']
urls=['https://genius.com/artists/Mc-lars', 'https://genius.com/artists/Johnson-and-haggkvist']
urls=['https://genius.com/artists/Rais', 'https://genius.com/artists/Metrickz', 'https://genius.com/artists/Omg-deu',
     'https://genius.com/artists/Almklausi', 'https://genius.com/artists/Ost-boys', 'https://genius.com/artists/Bonez-mc-and-raf-camora',
     'https://genius.com/artists/Lsp', 'https://genius.com/artists/Daniil', 'https://genius.com/artists/Amal-dnk', 'https://genius.com/artists/Billie-marten',
     'https://genius.com/artists/Robin-packalen', 'https://genius.com/artists/Ibe']
urls=['https://genius.com/artists/Hleb', 'https://genius.com/artists/25-17']
urls=['https://genius.com/artists/Ke-personajes', 'https://genius.com/artists/Tiago-pzk', 'https://genius.com/artists/Jencarlos',
      'https://genius.com/artists/Kilate-tesla', 'https://genius.com/artists/Chano', 'https://genius.com/artists/Panteon-rococo',
      'https://genius.com/artists/Chichi-peralta', 'https://genius.com/artists/Fito-paez', 'https://genius.com/artists/Death-cab-for-cutie']
urls=['https://genius.com/artists/Rap-viet', 'https://genius.com/artists/Fellow-fellow', 'https://genius.com/artists/Jovislash',
     'https://genius.com/artists/Sabrina-carpenter', 'https://genius.com/artists/Kostromin', 'https://genius.com/artists/Pyrokinesis',
     'https://genius.com/artists/Rose', 'https://genius.com/artists/Genius-romanizations']
urls=['https://genius.com/artists/Mace', 'https://genius.com/artists/Izi', 'https://genius.com/artists/Polo-g', 'https://genius.com/artists/Juice-wrld-and-marshmello',
     'https://genius.com/artists/Starboi3', 'https://genius.com/artists/Playboi-carti', 'https://genius.com/artists/Lin-manuel-miranda']
urls=['https://genius.com/artists/Migos', 'https://genius.com/artists/Calvin-harris', 'https://genius.com/artists/Dream-and-pmbata',
      'https://genius.com/artists/Carlo-anthony']
urls=['https://genius.com/artists/Ltd', 'https://genius.com/artists/Bo', 'https://genius.com/artists/Enchantment', 
      'https://genius.com/artists/The-umcs', 'https://genius.com/artists/Marty', 'https://genius.com/artists/Donnie-trumpet-and-the-social-experiment', 
      'https://genius.com/artists/King-bach', 'https://genius.com/artists/Magoo', 'https://genius.com/artists/Kevin-gates', 
      'https://genius.com/artists/Sugarhill-gang', 'https://genius.com/artists/Buddy', 'https://genius.com/artists/Yazz', 
      'https://genius.com/artists/Francis-and-the-lights', 'https://genius.com/artists/For-king-and-country-and-dolly-parton', 
      'https://genius.com/artists/Lovkn']
urls=['https://genius.com/artists/The-1975', 'https://genius.com/artists/Steve-oliver', 'https://genius.com/artists/Najee']
urls=['https://genius.com/artists/Hector-el-father']
urls=['https://genius.com/artists/Tone']
urls=['https://genius.com/artists/Epic-the-future', 'https://genius.com/artists/Rilo-kiley', 'https://genius.com/artists/Monsta']
urls=['https://genius.com/artists/Lab', 'https://genius.com/artists/Loft', 'https://genius.com/artists/Ida-madsen',
     'https://genius.com/artists/Rjan-nilsen', 'https://genius.com/artists/Emmy-emma-bejanyan']
urls=['https://genius.com/artists/Antytila', 'https://genius.com/artists/Sandy-and-junior', 'https://genius.com/artists/Lomepal',
     'https://genius.com/artists/Stoney-larue', 'https://genius.com/artists/Morgan-wallen', 'https://genius.com/artists/Hardy',
     'https://genius.com/artists/Sabrina-carpenter', 'https://genius.com/artists/The-longest-johns', 'https://genius.com/artists/Rick-astley',
     'https://genius.com/artists/Pharaoh', 'https://genius.com/artists/Brent-faiyaz-and-dj-dahi', 'https://genius.com/artists/Tyler-the-creator',
     'https://genius.com/artists/Odd-future', 'https://genius.com/artists/Travis-scott', 'https://genius.com/artists/Dusty-locane',
     'https://genius.com/artists/Pooh-shiesty', 'https://genius.com/artists/Lil-durk', 'https://genius.com/artists/Meek-mill',
     'https://genius.com/artists/Xxxtentacion', 'https://genius.com/artists/Markul', 'https://genius.com/artists/Dusty-locane',
     'https://genius.com/artists/Palc', 'https://genius.com/artists/Morgenshtern', 'https://genius.com/artists/Pop-smoke',
     'https://genius.com/artists/Bedoes-and-lanek', 'https://genius.com/artists/Pooh-shiesty', 'https://genius.com/artists/Blac-youngsta',
     'https://genius.com/artists/Elyotto', 'https://genius.com/artists/Slowthai', 'https://genius.com/artists/Amine', 'https://genius.com/artists/2pac']
urls=['https://genius.com/artists/Kiara']
for url in urls:
    print(url)
    dp = dbArtistsGenius()
    dp.downloadArtistFromURL(url)
    sleep(2)
#artistID = dp.dutils.getArtistID(url)
#savename = dp.dutils.getArtistSavename(artistID)
#print(url,savename)

# Download Search Data (Getting Artist IDs)

In [ ]:
from glob import glob
ts = timestat("Finding Searched For Artists")
searchedForArtist = Series([fileUtil(ifile).basename for ifile in glob("genius/search/*.p")], dtype = 'object')
ts.stop()
print("Found {0} Searched For Artists".format(len(searchedForArtist)))

In [ ]:
ignores="""
""".split("\n")
ignores = Series([x for x in ignores if len(x) > 0], dtype = 'object')

In [ ]:
from fsUtils import fileUtil
from time import sleep
from masterArtistNameCorrection import masterArtistNameCorrection
ts = timestat("Getting Artists To Download")
meu   = masterManualEntriesUtils()
mmeDF = meu.getDF()
manc = masterArtistNameCorrection()
artistNames = mmeDF["ArtistName"].apply(manc.clean)
print("Found {0} Artists".format(artistNames.shape[0]))
artistNamesToDownload = artistNames[~artistNames.isin(searchedForArtist)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
artistNamesToDownload = artistNamesToDownload[~artistNamesToDownload.isin(ignores)]
print("Found {0} Artists To Download".format(artistNamesToDownload.shape[0]))
ts.stop()

In [ ]:
toget = {}
from masterArtistNameCorrection import masterArtistNameCorrection
manc = masterArtistNameCorrection()
for i,(idx,artistName) in enumerate(artistNamesToDownload.iteritems()):
    if i >= 5000:
        break
    savefile = dirUtil(dirUtil("genius").join("search")).join("{0}.p".format(manc.clean(artistName)))
    if savefile.exists():
        continue
    toget[savefile] = artistName
ts.stop()
print("Need To Download {0} Artists".format(len(toget)))

In [ ]:
ts = timestat("Downloading {0} Genius Artist Data".format(len(toget)))
nErr = 0
for i,(savefile,artistName) in enumerate(toget.items()):
    if nErr >= 5:
        break
    result = getSearchResults(artistName)
    if result is None:
        print("Error with {0}".format(artistName))
        nErr += 1
        sleep(60*nErr)
        continue
    io.save(idata=result, ifile=savefile)
    sleep(4.0)
    
    if (i+1) % 25 == 0:
        print("")
        ts.update(n=i+1,N=len(toget))
        sleep(1.5)
        print("")
ts.stop()

# Get Artist IDs From Web Pages

# Organize Search Results

In [ ]:
from glob import glob
files = glob("genius/ModVal*.p")
artists = []
subs    = []
media   = []
lyrics  = []
for i,ifile in enumerate(files):
    modData = io.get(ifile)
    for item in modData:
        artists.append(item["PrimaryArtist"])
        for artist in item["SubArtists"]:
            subs.append(artist)
        for album in item["Albums"]:
            media.append(album['Artist'])
            lyrics.append({"id": album["LyricsArtistID"], 'title': album['Title']})
        for song in item["Songs"]:
            media.append(song['Artist'])
            
    if (i+1) % 5 == 0:
        print("{0: <6}{1: <8}{2: <8}".format(i+1,len(artists),len(lyrics)))

In [ ]:
from pandas import concat

artistDF = DataFrame(artists)
mediaDF  = DataFrame(media)
subsDF   = DataFrame(subs)
lyricsDF = DataFrame(lyrics)

genIDs = concat([artistDF['id'], mediaDF['id'], subsDF['id'], lyricsDF['id']])
genIDCounts = genIDs.value_counts().sort_values(ascending=False)

artistsDF = artistDF[~artistDF['id'].duplicated()]
mediaDF   = mediaDF[~mediaDF['id'].duplicated()]
subsDF    = subsDF[~subsDF['id'].duplicated()]
lyricsDF  = lyricsDF[~lyricsDF['id'].duplicated()]

artistsDF = artistsDF.append(mediaDF[~mediaDF['id'].isin(artistsDF['id'])])
artistsDF = artistsDF.append(subsDF[~subsDF['id'].isin(artistsDF['id'])])
#artistsDF = artistsDF.append(lyricsDF[~lyricsDF['id'].isin(artistsDF['id'])])

cnts = artistsDF['id'].apply(genIDCounts.get)
cnts.name = "Counts"

artistsDF.index = artistsDF['id']
artistsDF.drop(['id'], axis=1, inplace=True)
artistsDF = artistsDF.join(cnts)
artistsDF['Counts'] = artistsDF['Counts'].fillna(0).astype(int)

artistsDF = artistsDF.sort_values(by="Counts", ascending=False)

In [ ]:
io.save(idata=artistsDF, ifile="geniusArtistRanking.p")

# Download Artist API Data

# Search For Missing Artists

In [ ]:
from glob import glob
from pandas import concat
files = glob("genius/search/*.p")
primaryList = []
lyricsList  = []
ts = timestat("Creating Primary/Lyrics Data From {0} Files".format(len(files)))
for i,ifile in enumerate(files):
    df = DataFrame([{"id": item['artist']['id'], "counts": item['pyongs_cnt'], 'lyrics': item['lyrics_id']} for idx,item in io.get(ifile).iteritems()])
    
    primary = df.groupby('id').agg({"counts": "sum"})
    primary["artistName"] = fileUtil(ifile).basename
    primaryList.append(primary)
    lyrics  = df.groupby('lyrics').agg({"counts": "sum"})
    lyrics["artistName"] = fileUtil(ifile).basename
    lyricsList.append(lyrics)
    
    if (i+1) % 100 == 0:
        ts.update(n=i+1,N=len(files))
    
primary = concat(primaryList)
primary = primary.sort_values(by="counts", ascending=False)

lyrics = concat(lyricsList)
lyrics = lyrics.sort_values(by="counts", ascending=False)
ts.stop()